In [ ]:
n = int(input())
e = n
o = n
while e > 10 or o > 10:
    ect = 0
    oct = 0
    if e > 10:
        for x in str(e):
            if int(x) % 2 == 0:
                ect += int(x)
            else:
                oct += int(x)
    else:
        ect = e
    if o > 10:
        for x in str(o):
            if int(x) % 2 != 0:
                oct += int(x)
            else:
                ect += int(x)
    else:
        oct = o
    e = ect
    o = oct
print(e, o)

2 9


In [ ]:
# ==========================================
# IMPORT REQUIRED MODULES
# ==========================================

import sqlite3 as sql                     # SQLite database module
from datetime import datetime            # For current date and time

# ==========================================
# DATABASE CONNECTION
# ==========================================

# Connect to SQLite database
conn = sql.connect('example.db')

# Create cursor object
c = conn.cursor()

# ==========================================
# CREATE TABLES
# ==========================================

# Create main bank table
c.execute('''
CREATE TABLE IF NOT EXISTS hindhu_bank (
    name TEXT,
    balance REAL,
    password TEXT,
    phone TEXT PRIMARY KEY,
    email TEXT
)
''')

# Create transaction history table
c.execute('''
CREATE TABLE IF NOT EXISTS transactions (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    phone TEXT,
    type TEXT,
    amount REAL,
    date TEXT
)
''')

# Create loan table
c.execute('''
CREATE TABLE IF NOT EXISTS loans (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    phone TEXT,
    amount REAL,
    date TEXT
)
''')

# Create users login table
c.execute('''
CREATE TABLE IF NOT EXISTS users (
    phone TEXT PRIMARY KEY,
    password TEXT
)
''')

# Create admin table
c.execute('''
CREATE TABLE IF NOT EXISTS admins (
    username TEXT PRIMARY KEY,
    password TEXT
)
''')

# Save all table changes
conn.commit()

# ==========================================
# HELPER FUNCTIONS
# ==========================================

# Store transaction details
def transaction_log(phone, t_type, amount):

    # Get current date and time
    date = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    # Insert transaction record
    c.execute(
        "INSERT INTO transactions (phone, type, amount, date) VALUES (?, ?, ?, ?)",
        (phone, t_type, amount, date)
    )

    # Save changes
    conn.commit()


# Check whether account exists
def account_exists(phone):

    # Search account using phone number
    c.execute("SELECT * FROM hindhu_bank WHERE phone=?", (phone,))

    # Return True if found
    return c.fetchone() is not None


# ==========================================
# USER FUNCTIONS
# ==========================================

# Register new user
def register_user(phone, password):

    try:

        # Insert new user
        c.execute(
            "INSERT INTO users (phone, password) VALUES (?, ?)",
            (phone, password)
        )

        # Save changes
        conn.commit()

        print("User registered successfully")

    except sql.IntegrityError:

        # If user already exists
        print("User already exists")


# Register admin
def register_admin(username, password):

    try:

        # Insert admin details
        c.execute(
            "INSERT INTO admins (username, password) VALUES (?, ?)",
            (username, password)
        )

        # Save changes
        conn.commit()

        print("Admin registered successfully")

    except sql.IntegrityError:

        # If admin already exists
        print("Admin already exists")


# User login verification
def login_user(phone, password):

    # Check matching phone and password
    c.execute(
        "SELECT * FROM users WHERE phone=? AND password=?",
        (phone, password)
    )

    # Return login status
    return c.fetchone() is not None


# Admin login verification
def login_admin(username, password):

    # Check admin credentials
    c.execute(
        "SELECT * FROM admins WHERE username=? AND password=?",
        (username, password)
    )

    # Return login result
    return c.fetchone() is not None


# Logout function
def logout():

    print("Logged out successfully")


# ==========================================
# ACCOUNT FUNCTIONS
# ==========================================

# Create new bank account
def create_account(name, balance, password, phone, email):

    # Check duplicate account
    if account_exists(phone):

        print("Account already exists")
        return

    # Insert account details
    c.execute(
        '''
        INSERT INTO hindhu_bank
        (name, balance, password, phone, email)
        VALUES (?, ?, ?, ?, ?)
        ''',
        (name, balance, password, phone, email)
    )

    # Save changes
    conn.commit()

    # Create login credentials
    register_user(phone, password)

    # Store transaction
    transaction_log(phone, "Account Created", balance)

    print("Account created successfully")


# Delete account
def delete_account(phone):

    # Check account exists
    if not account_exists(phone):

        print("Account not found")
        return

    # Delete account details
    c.execute("DELETE FROM hindhu_bank WHERE phone=?", (phone,))

    # Delete login details
    c.execute("DELETE FROM users WHERE phone=?", (phone,))

    # Save changes
    conn.commit()

    print("Account deleted successfully")


# Update account balance
def update_balance(phone, amount):

    # Check account exists
    if not account_exists(phone):

        print("Account not found")
        return

    # Add or subtract balance
    c.execute(
        "UPDATE hindhu_bank SET balance = balance + ? WHERE phone=?",
        (amount, phone)
    )

    # Save changes
    conn.commit()


# Get balance amount
def get_balance(phone):

    # Fetch balance
    c.execute(
        "SELECT balance FROM hindhu_bank WHERE phone=?",
        (phone,)
    )

    # Store fetched data
    data = c.fetchone()

    # If account found
    if data:

        return data[0]

    # If account not found
    return None


# Get account details
def get_account_details(phone):

    # Fetch account details
    c.execute(
        "SELECT name, balance, phone, email FROM hindhu_bank WHERE phone=?",
        (phone,)
    )

    return c.fetchone()


# Change password
def change_password(phone, new_password):

    # Check account exists
    if not account_exists(phone):

        print("Account not found")
        return

    # Update bank table password
    c.execute(
        "UPDATE hindhu_bank SET password=? WHERE phone=?",
        (new_password, phone)
    )

    # Update login table password
    c.execute(
        "UPDATE users SET password=? WHERE phone=?",
        (new_password, phone)
    )

    # Save changes
    conn.commit()

    print("Password changed successfully")


# ==========================================
# BANKING OPERATIONS
# ==========================================

# Transfer money between accounts
def transfer_funds(from_phone, to_phone, amount):

    # Check sender account
    if not account_exists(from_phone):

        print("Sender account not found")
        return

    # Check receiver account
    if not account_exists(to_phone):

        print("Receiver account not found")
        return

    # Get sender balance
    balance = get_balance(from_phone)

    # Check sufficient balance
    if balance >= amount:

        # Deduct sender balance
        update_balance(from_phone, -amount)

        # Add receiver balance
        update_balance(to_phone, amount)

        # Store sender transaction
        transaction_log(from_phone, "Transfer Sent", amount)

        # Store receiver transaction
        transaction_log(to_phone, "Transfer Received", amount)

        print("Transfer successful")

    else:

        print("Insufficient funds")


# Withdraw money
def withdraw(phone, amount):

    # Check account exists
    if not account_exists(phone):

        print("Account not found")
        return

    # Get current balance
    balance = get_balance(phone)

    # Check sufficient balance
    if balance >= amount:

        # Deduct amount
        update_balance(phone, -amount)

        # Store transaction
        transaction_log(phone, "Withdraw", amount)

        print("Withdrawal successful")

    else:

        print("Insufficient funds")


# Deposit money
def deposit(phone, amount):

    # Check account exists
    if not account_exists(phone):

        print("Account not found")
        return

    # Add amount
    update_balance(phone, amount)

    # Store transaction
    transaction_log(phone, "Deposit", amount)

    print("Deposit successful")


# Display balance
def check_balance(phone):

    # Get balance
    balance = get_balance(phone)

    # If account exists
    if balance is not None:

        print("Available Balance:", balance)

    else:

        print("Account not found")


# View all transactions
def statements(phone):

    # Fetch transaction records
    c.execute(
        "SELECT type, amount, date FROM transactions WHERE phone=?",
        (phone,)
    )

    # Store all records
    data = c.fetchall()

    # If transactions found
    if data:

        for row in data:

            print(row)

    else:

        print("No transactions found")


# Loan request function
def loan(phone, amount):

    # Check account exists
    if not account_exists(phone):

        print("Account not found")
        return

    # Get current date and time
    date = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    # Insert loan request
    c.execute(
        "INSERT INTO loans (phone, amount, date) VALUES (?, ?, ?)",
        (phone, amount, date)
    )

    # Save changes
    conn.commit()

    print("Loan request submitted successfully")


# ==========================================
# MAIN PROGRAM
# ==========================================

# Main menu function
def main():

    print("===== Welcome to Hindhu Bank =====")

    # Infinite loop
    while True:

        # Display menu
        print("\n========= MENU =========")

        print("1. Create Account")
        print("2. Delete Account")
        print("3. Get Balance")
        print("4. Account Details")
        print("5. Change Password")
        print("6. Transfer Funds")
        print("7. Withdraw")
        print("8. Deposit")
        print("9. Check Balance")
        print("10. Statements")
        print("11. Apply Loan")
        print("12. Exit")

        # Get user choice
        choice = input("Enter your choice: ")

        try:

            # Create account
            if choice == '1':

                name = input("Enter Name: ")
                balance = float(input("Enter Initial Balance: "))
                password = input("Enter Password: ")
                phone = input("Enter Phone: ")
                email = input("Enter Email: ")

                create_account(
                    name,
                    balance,
                    password,
                    phone,
                    email
                )

            # Delete account
            elif choice == '2':

                phone = input("Enter Phone: ")

                delete_account(phone)

            # Get balance
            elif choice == '3':

                phone = input("Enter Phone: ")

                print("Balance:", get_balance(phone))

            # Get account details
            elif choice == '4':

                phone = input("Enter Phone: ")

                print(get_account_details(phone))

            # Change password
            elif choice == '5':

                phone = input("Enter Phone: ")
                new_password = input("Enter New Password: ")

                change_password(phone, new_password)

            # Transfer funds
            elif choice == '6':

                from_phone = input("Sender Phone: ")
                to_phone = input("Receiver Phone: ")
                amount = float(input("Enter Amount: "))

                transfer_funds(from_phone, to_phone, amount)

            # Withdraw money
            elif choice == '7':

                phone = input("Enter Phone: ")
                amount = float(input("Enter Amount: "))

                withdraw(phone, amount)

            # Deposit money
            elif choice == '8':

                phone = input("Enter Phone: ")
                amount = float(input("Enter Amount: "))

                deposit(phone, amount)

            # Check balance
            elif choice == '9':

                phone = input("Enter Phone: ")

                check_balance(phone)

            # View statements
            elif choice == '10':

                phone = input("Enter Phone: ")

                statements(phone)

            # Apply loan
            elif choice == '11':

                phone = input("Enter Phone: ")
                amount = float(input("Enter Loan Amount: "))

                loan(phone, amount)

            # Exit program
            elif choice == '12':

                print("Thank you for using Hindhu Bank")

                break

            # Invalid choice
            else:

                print("Invalid choice")

        # Invalid numeric input
        except ValueError:

            print("Invalid numeric input")

        # Other errors
        except Exception as e:

            print("Error:", e)


# ==========================================
# START PROGRAM
# ==========================================

# Run main function
if __name__ == "__main__":

    main()

    # Close database connection
    conn.close()

In [2]:
n=int(input("enter number of elements: "))
l=[]
for x in range(n):
    l.append(int(input("enter element: ")))
t=int(input("enter the target value: "))
for i in range(n-1):
    for j in range(i+1,n):
        if l[i]+l[j]==t:
            print("pair found: ",l[i],l[j])

pair found:  1 4
pair found:  2 3


In [ ]:
l=[1,1,1,1,2,2,3,3,3,4,4,5,4,6,6,5,5,3,4,5,3,4,5]
for x in set(l):
    print(x, l.count(x))

1 4
2 2
3 5
4 5
5 5
6 2


In [2]:
# 1) list of unique values and their counts, without using set() and count() functions, using lambda function
l=[1,1,1,1,2,2,3,3,3,4,4,5,4,6,6,5,5,3,4,5,3,4,5]

unique_values = list(filter(lambda x: l.count(x) > 0, set(l)))
list(map(lambda x: print(x, l.count(x)), unique_values))

1 4
2 2
3 5
4 5
5 5
6 2


[None, None, None, None, None, None]

In [ ]:
1